# permute-back-argsort — ex2: permute round-trip invariant — verify forward∘backward = identity

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `permute-back-argsort`. Running the final beacon cell reports progress against the `Backprop: permute_back via argsort` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: permute_back via argsort` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`permute-back-argsort`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "permute-back-argsort"
DD_SUBTOPIC = "Backprop: permute_back via argsort"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `permute_back` round-trip invariant — quick refresher

ex1 derived `inverse = argsort(dims)`. The deeper facet is the STRUCTURAL invariant this produces: for ANY valid permutation `dims`, applying `permute_back` to `grad_out = x.permute(*dims)` returns `x` itself.

```
x.permute(*dims).permute(*argsort(dims)) == x   # for every dims
```

Why this matters: if you forward then immediately backward through a permute, the gradient flowing into `x` is identical to the gradient that exited `out` — because `permute` is a pure axis shuffle with no value change, the backward is a pure inverse shuffle.

Pinning the round-trip identity gives you a TEST that catches inverse-permutation bugs without needing to derive the gradient by hand.

### Exercise 2 — permute round-trip invariant — verify forward∘backward = identity

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the structural invariant: x.permute(*dims).permute(*argsort(dims)) == x for every valid permutation, and use it to verify permute_back.
> Keywords: permute, round-trip, invariant, argsort, structural
> ```

**KCs targeted:** `permute-backward-pattern`, `inverse-permutation-via-argsort`

Implement TWO pieces:

1. **`permute_back(grad_out, out, x, dims)`** — apply the inverse permutation via `argsort`.

2. **`assert_permute_round_trip(x, dims)`** — a property checker. Compute `forward = x.permute(*dims)`, then `back_to_x = permute_back(forward, forward, x, dims)`. Assert `back_to_x.shape == x.shape` AND `t.equal(back_to_x, x)` (bit-exact — permute does no arithmetic). Raise `AssertionError` with a message naming `dims` on failure.

**Why analyze rather than apply.** ex1 had you derive the back fn. ex2 makes you EXAMINE the structural property that derivation produces: applying `permute(*dims)` then `permute(*argsort(dims))` is a NO-OP for ANY valid `dims`. Pinning this invariant in code gives you a property-based test that catches inverse-permutation bugs WITHOUT having to derive the gradient by hand on a case-by-case basis.

The test runs `assert_permute_round_trip` against every permutation of `(0, 1, 2)` and several 4-D permutations.

No autograd.

In [ ]:
def permute_back(grad_out: Tensor, out: Tensor, x: Tensor, dims: tuple) -> Tensor:
    inverse = tuple(int(i) for i in np.argsort(dims))
    return grad_out.permute(*inverse)


def assert_permute_round_trip(x: Tensor, dims: tuple) -> None:
    forward = x.permute(*dims)
    back_to_x = permute_back(forward, forward, x, dims)
    assert back_to_x.shape == x.shape, (
        f'shape broke on dims={dims}: got {back_to_x.shape}, expected {x.shape}'
    )
    assert t.equal(back_to_x, x), (
        f'round-trip on dims={dims} failed: max diff {(back_to_x - x).abs().max()}'
    )


<details><summary>Solution</summary>

```python
def permute_back(grad_out: Tensor, out: Tensor, x: Tensor, dims: tuple) -> Tensor:
    inverse = tuple(int(i) for i in np.argsort(dims))
    return grad_out.permute(*inverse)


def assert_permute_round_trip(x: Tensor, dims: tuple) -> None:
    forward = x.permute(*dims)
    back_to_x = permute_back(forward, forward, x, dims)
    assert back_to_x.shape == x.shape, (
        f'shape broke on dims={dims}: got {back_to_x.shape}, expected {x.shape}'
    )
    assert t.equal(back_to_x, x), (
        f'round-trip on dims={dims} failed: max diff {(back_to_x - x).abs().max()}'
    )
```

**Why bit-exact equality, not allclose.** Permute reads from storage by stride — no arithmetic — so the values are identical bit patterns. `t.equal` is the right pin; `t.allclose` would mask permutation bugs that happen to produce nearby float values.

**Why this is Analyze-Bloom.** ex1 was Apply: 'compute the inverse'. ex2 is Analyze: 'identify the structural property that MAKES the inverse correct'. The round-trip invariant is what lets you trust `argsort` without rederiving on every shape — and what catches monkey-patched bugs (as the test demonstrates).

**Property-based testing in autograd.** Real frameworks (PyTorch, JAX) include round-trip tests like this in their CI — exactly because they catch implementation bugs that per-case assertion tests can miss. The pattern transfers.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()